# Brain Tumor MRI Adversarial Robustness

Research notebook for four-class brain tumor MRI classification using VGG16, FGSM and PGD adversarial attacks, adversarial training, and robustness evaluation.

This notebook contains genuine experimental code from the brain tumor adversarial-robustness research workflow. It is shared as a supporting research notebook. Additional materials associated with the broader study may be requested when available and shareable.

**Data:** Kaggle Brain Tumor MRI Dataset (`masoudnickparvar/brain-tumor-mri-dataset`). Do not commit `kaggle.json`, API keys, datasets, or model checkpoints to a public repository.


In [ ]:
# Step 1: Upload Kaggle API key (kaggle.json)
from google.colab import files
files.upload()
# Step 2: Setup Kaggle Directory and Permissions
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Step 3: Download Dataset Using Kaggle API
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p ./brain_tumor_data --unzip

# Step 4: Load the Data for Analysis
import os
import matplotlib.pyplot as plt
import cv2

data_dir = './brain_tumor_data/Training'
categories = os.listdir(data_dir)
print(f"Categories: {categories}")

# Step 5: View Some of the Images
for category in categories:
    path = os.path.join(data_dir, category)
    sample_images = os.listdir(path)[:5]  # Get the first 5 images from each category

    for img_name in sample_images:
        img_path = os.path.join(path, img_name)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert from BGR to RGB for correct color representation

        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.title(f"Category: {category}")
        plt.axis('off')
        plt.show()

In [ ]:
# Import Libraries
import os
import numpy as np
import tensorflow as tf
import cv2
import random
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import VGG16
from sklearn.metrics import classification_report, confusion_matrix

# Image Processing
IMG_SIZE = 128

def preprocess_images(data_directory, categories, img_size):
    data = []
    for category in categories:
        path = os.path.join(data_directory, category)
        class_label = categories.index(category)
        for img_name in os.listdir(path):
            try:
                img_path = os.path.join(path, img_name)
                img = cv2.imread(img_path, cv2.IMREAD_COLOR)
                if img is not None:
                    img = cv2.resize(img, (img_size, img_size))
                    img = img / 255.0
                    data.append([img, class_label])
            except Exception as e:
                print(f"Error loading image {img_name}: {e}")
    return data

# Load images
data_dir = './brain_tumor_data/Training'  # Your dataset path
categories = os.listdir(data_dir)
categories = [category for category in categories if os.path.isdir(os.path.join(data_dir, category))]
train_data = preprocess_images(data_dir, categories, IMG_SIZE)

# Prepare data
random.shuffle(train_data)
X, y = zip(*train_data)
X, y = np.array(X), np.array(y)

# Split data
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Definition
base_model = VGG16(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(categories), activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Initial training
initial_epochs = 5
model.fit(X_train, y_train, epochs=initial_epochs, validation_data=(X_val, y_val))

# Define FGSM and PGD Attack Functions
def fgsm_attack(model, images, labels, epsilon=0.01):
    images = tf.convert_to_tensor(images, dtype=tf.float32)
    labels = tf.convert_to_tensor(labels, dtype=tf.int64)
    with tf.GradientTape() as tape:
        tape.watch(images)
        predictions = model(images)
        loss = tf.keras.losses.sparse_categorical_crossentropy(labels, predictions)
    gradients = tape.gradient(loss, images)
    perturbations = epsilon * tf.sign(gradients)
    adv_images = images + perturbations
    return tf.clip_by_value(adv_images, 0, 1).numpy()

def pgd_attack(model, images, labels, epsilon=0.01, alpha=0.002, num_iter=3):
    adv_images = tf.convert_to_tensor(images, dtype=tf.float32)
    labels = tf.convert_to_tensor(labels, dtype=tf.int64)
    for i in range(num_iter):
        with tf.GradientTape() as tape:
            tape.watch(adv_images)
            predictions = model(adv_images)
            loss = tf.keras.losses.sparse_categorical_crossentropy(labels, predictions)
        gradients = tape.gradient(loss, adv_images)
        adv_images = adv_images + alpha * tf.sign(gradients)
        perturbation = tf.clip_by_value(adv_images - images, -epsilon, epsilon)
        adv_images = tf.clip_by_value(images + perturbation, 0, 1)
    return adv_images.numpy()

# Generate adversarial training data
X_train_adv_fgsm = fgsm_attack(model, X_train[:200], y_train[:200], epsilon=0.01)
X_train_adv_pgd = pgd_attack(model, X_train[:200], y_train[:200], epsilon=0.01, alpha=0.002, num_iter=3)
X_train_combined = np.concatenate((X_train[:200], X_train_adv_fgsm, X_train_adv_pgd))
y_train_combined = np.concatenate((y_train[:200], y_train[:200], y_train[:200]))

# Fine-tune model on combined clean and adversarial data
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train_combined, y_train_combined, epochs=5, validation_data=(X_val, y_val))

# Evaluation functions
def evaluate_model(model, X, y, categories, title='Model Evaluation'):
    predictions = model.predict(X)
    predicted_labels = np.argmax(predictions, axis=1)
    print(title)
    print(classification_report(y, predicted_labels, target_names=categories))
    print(confusion_matrix(y, predicted_labels))
    return predictions

# Evaluate on clean validation set
clean_predictions = evaluate_model(model, X_val, y_val, categories, 'Clean Validation Evaluation')

# Evaluate FGSM robustness
X_val_fgsm = fgsm_attack(model, X_val, y_val, epsilon=0.01)
fgsm_predictions = evaluate_model(model, X_val_fgsm, y_val, categories, 'FGSM Evaluation')

# Evaluate PGD robustness
X_val_pgd = pgd_attack(model, X_val, y_val, epsilon=0.01, alpha=0.002, num_iter=3)
pgd_predictions = evaluate_model(model, X_val_pgd, y_val, categories, 'PGD Evaluation')

In [ ]:
# Import necessary libraries for evaluation and plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Helper function for plotting training history
def plot_training_history(history, title='Training and Validation Metrics'):
    plt.figure(figsize=(14, 5))

    # Plot accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    plt.grid(True)

    # Plot loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)

    plt.suptitle(title)
    plt.show()

# Step 4: Confidence Bar Plot Function
def plot_confidence_bars_with_images(predictions, true_labels, categories, images, title, limit=5):
    indices = np.random.choice(len(true_labels), size=limit, replace=False)
    predicted_labels = np.argmax(predictions, axis=1)[indices]
    true_labels = np.array(true_labels)[indices]
    images = np.array(images)[indices]

    fig, axes = plt.subplots(len(true_labels), 3, figsize=(10, 3 * len(true_labels)), constrained_layout=True)
    color_palette = sns.color_palette("hsv", len(categories))

    for i, (pred, true, img) in enumerate(zip(predicted_labels, true_labels, images)):
        axes[i][0].imshow(img)
        axes[i][0].set_title(f'Original: {categories[true]}')
        axes[i][0].axis('off')

        axes[i][1].imshow(img)
        axes[i][1].set_title(f'Predicted: {categories[pred]} ({predictions[indices[i]][pred]:.2f})')
        axes[i][1].axis('off')

        bars = axes[i][2].barh(categories, predictions[indices[i]], color=color_palette, edgecolor='black', zorder=3)
        for bar in bars:
            bar.set_linewidth(1.0)
        axes[i][2].set_xlabel('Confidence (%)')
        axes[i][2].grid(True, linestyle='--', alpha=0.8, zorder=1)
        axes[i][2].set_xlim(0, 1)
        axes[i][2].set_title(title)

    plt.show()

# Initial Training and Adversarial Training Phases
initial_epochs = 5
history_initial = model.fit(X_train, y_train, epochs=initial_epochs, validation_data=(X_val, y_val))

# Plot training history after initial training
plot_training_history(history_initial, title='Initial Training Metrics')

# FGSM and PGD Adversarial Training Examples
X_train_adv_fgsm = fgsm_attack(model, X_train[:200], y_train[:200], epsilon=0.01)
X_train_adv_pgd = pgd_attack(model, X_train[:200], y_train[:200], epsilon=0.01, alpha=0.002, num_iter=3)
X_train_combined = np.concatenate((X_train[:200], X_train_adv_fgsm, X_train_adv_pgd))
y_train_combined = np.concatenate((y_train[:200], y_train[:200], y_train[:200]))

# Fine-tune model on combined clean and adversarial data
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_adv = model.fit(X_train_combined, y_train_combined, epochs=5, validation_data=(X_val, y_val))

# Plot adversarial training history
plot_training_history(history_adv, title='Adversarial Training Metrics')

# Clean validation evaluation
clean_predictions = model.predict(X_val)
clean_labels = np.argmax(clean_predictions, axis=1)
print('Clean Validation Classification Report')
print(classification_report(y_val, clean_labels, target_names=categories))

# FGSM evaluation
X_val_fgsm = fgsm_attack(model, X_val, y_val, epsilon=0.01)
fgsm_predictions = model.predict(X_val_fgsm)
fgsm_labels = np.argmax(fgsm_predictions, axis=1)
print('FGSM Classification Report')
print(classification_report(y_val, fgsm_labels, target_names=categories))

# PGD evaluation
X_val_pgd = pgd_attack(model, X_val, y_val, epsilon=0.01, alpha=0.002, num_iter=3)
pgd_predictions = model.predict(X_val_pgd)
pgd_labels = np.argmax(pgd_predictions, axis=1)
print('PGD Classification Report')
print(classification_report(y_val, pgd_labels, target_names=categories))

# Confusion matrices
def plot_confusion(cm, categories, title):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=categories, yticklabels=categories)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(title)
    plt.show()

plot_confusion(confusion_matrix(y_val, clean_labels), categories, 'Clean Validation Confusion Matrix')
plot_confusion(confusion_matrix(y_val, fgsm_labels), categories, 'FGSM Confusion Matrix')
plot_confusion(confusion_matrix(y_val, pgd_labels), categories, 'PGD Confusion Matrix')

# Prediction confidence visualizations
plot_confidence_bars_with_images(clean_predictions, y_val, categories, X_val, 'Clean Prediction Confidence', limit=5)
plot_confidence_bars_with_images(fgsm_predictions, y_val, categories, X_val_fgsm, 'FGSM Prediction Confidence', limit=5)
plot_confidence_bars_with_images(pgd_predictions, y_val, categories, X_val_pgd, 'PGD Prediction Confidence', limit=5)